# Crisis Connect — Phase 3a: Audio + Text Preprocessing
**CSC-233 Artificial Intelligence Lab | Spring 2026**
**Component owner: Imman Aamir**

### Classes: earthquake | flood | fire | traffic_incident

**Run order:**
- Cell 1 ✅ Install packages
- Cell 2 ✅ Mount Drive
- Cell 3 ✅ Load Whisper
- Cell 4 ✅ Generate synthetic text
- Cell 5 ✅ Transcribe Urdu audio (NOW ACTIVE)
- Cell 6 ✅ Clean and merge all text
- Cell 7 ✅ TF-IDF vectorisation
- Cell 8 ✅ Split and save
- Cell 9 ✅ Summary

## Cell 1 — Install packages

In [ ]:
!pip install openai-whisper scikit-learn nltk deep-translator -q
!apt-get install -y ffmpeg -q

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

print('All packages installed.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 18.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 752.1 kB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.
All packages installed.


## Cell 2 — Mount Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

CLASSES    = ['earthquake', 'flood', 'fire', 'traffic_incident']
OUTPUT_DIR = '/content/drive/MyDrive/crisis connect_phase3a'
AUDIO_DIR  = '/content/drive/MyDrive/crisis connect_audio'

os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Output will be saved to:', OUTPUT_DIR)
print()

# Check audio folders exist
print('Checking audio folders:')
for cls in CLASSES:
    folder = os.path.join(AUDIO_DIR, cls + ' audio')
    if os.path.exists(folder):
        files = [f for f in os.listdir(folder) if f.lower().endswith(('.m4a','.mp3','.wav','.ogg','.flac'))]
        print(f'  {cls + " audio":<30} {len(files)} files found')
    else:
        print(f'  {cls + " audio":<30} NOT FOUND')

Mounted at /content/drive
Output will be saved to: /content/drive/MyDrive/crisis connect_phase3a

Checking audio folders:
  earthquake audio               6 files found
  flood audio                    6 files found
  fire audio                     6 files found
  traffic_incident audio         6 files found


## Cell 3 — Load Whisper model

In [ ]:
import whisper

whisper_model = whisper.load_model('small')

print('Whisper small model loaded.')
print('Supports Urdu, English, and code-switched speech.')

100%|████████████████████████████████████████| 461M/461M [00:03<00:00, 161MiB/s]


Whisper small model loaded.
Supports Urdu, English, and code-switched speech.


## Cell 4 — Generate synthetic text descriptions

In [ ]:
import pandas as pd
import random
import os
random.seed(42)

DESCRIPTIONS = {
    'earthquake': [
        'Strong shaking felt for about 30 seconds walls cracked',
        'Building collapsed after earthquake people trapped inside',
        'Earthquake just hit ground is still shaking slightly',
        'Cracks in the walls and ceiling after the tremor',
        'Several houses collapsed in the neighborhood after quake',
        'Aftershocks continuing people afraid to enter buildings',
        'Landslide triggered by earthquake blocking the main road',
        'Felt strong vibrations twice in the last ten minutes',
        'Roof of old school building fell during the earthquake',
        'People running out of buildings shouting earthquake',
        'Large cracks visible on the road surface after quake',
        'Multiple buildings damaged in the area after tremor',
        'Old mosque wall collapsed during the earthquake today',
        'Ground opened up near the fields after the quake',
        'Earthquake at night everyone left houses in fear',
        'Power lines fell after strong earthquake tremor here',
        'Earthquake felt from fifth floor very strong shaking',
        'Rocky area showing fresh landslide after earthquake',
        'Gas smell after earthquake possible pipe burst inside',
        'Strong earthquake damaged bridge near the river crossing',
        'Tremor lasted two minutes walls of house completely cracked',
        'Emergency services needed earthquake destroyed whole block',
        'Felt strong shaking everything fell off shelves',
        'Earthquake damaged water supply pipes area has no water',
        'People trapped under rubble after building collapsed in quake',
    ],
    'flood': [
        'Water level rising fast near the bridge on main road',
        'Houses submerged up to the roof in the village',
        'Road completely underwater cars stranded near market',
        'River has overflowed its banks near the town',
        'Flood water entering homes in the lower area',
        'People stranded on rooftops due to rising water',
        'Water level at school compound above knee height',
        'Flash flood coming down from the hills toward village',
        'Crops and fields completely submerged in flood water',
        'Flood destroyed the road no access to the village',
        'Water entering from the drain streets flooded',
        'River bank broken water rushing into residential area',
        'Flood water brown and fast moving along main street',
        'Children and elderly trapped inside flooded house',
        'Bridge submerged no way to cross to other side',
        'Water rising every hour need immediate evacuation help',
        'Hospital area flooded patients need to be moved',
        'Monsoon flood worse than last year in this area',
        'Water has reached the second floor of our building',
        'Entire street submerged rescue boats needed urgently',
        'Floodwater contaminated with sewage health emergency',
        'Canal overflowed flooding nearby homes and farms',
        'Rising waters cut off village from main city',
        'Livestock drowning in flood near the farm area',
        'Flood alert issued water approaching danger mark',
    ],
    'fire': [
        'Large fire in the market multiple shops burning now',
        'House on fire family trapped inside on upper floor',
        'Fire spreading from one building to the next quickly',
        'Black smoke visible from far fire near the factory',
        'Forest fire moving toward the village in the hills',
        'Fire truck arrived but water pressure is too low',
        'Gas cylinder exploded fire now spreading to neighbor',
        'Children inside the burning school building need help',
        'Electrical fire in the apartment smell of burning wire',
        'Wheat field on fire wind spreading it toward homes',
        'Fire at fuel station explosions heard from distance',
        'Smoke filling the street from burning warehouse nearby',
        'Fire started in kitchen and spread to whole house',
        'Large flames visible on hilltop forest fire growing',
        'Fire destroying old wooden market building rapidly',
        'Multiple vehicles burning after accident on highway',
        'Burning smell and smoke coming from the factory area',
        'Fire broke out in hospital storage room staff evacuating',
        'Whole block of shops on fire nothing left standing',
        'Fire started at night whole family woke up coughing',
        'Wildfire spreading fast due to strong winds nearby',
        'Chemical plant on fire toxic smoke spreading to area',
        'Fire engulfed three houses before brigade arrived',
        'Flames shooting from roof of residential building',
        'Market fire causing massive loss businesses destroyed',
    ],
    'traffic_incident': [
        'Major road accident multiple vehicles involved near town',
        'Truck overturned on highway blocking all lanes of traffic',
        'Head on collision between two buses many injured',
        'Car fell into ditch after losing control on wet road',
        'Motorbike accident on main road rider seriously injured',
        'Traffic jam due to accident ambulance cannot reach site',
        'Vehicle hit pedestrians at busy intersection downtown',
        'Passenger van fell into ravine on mountain road',
        'Oil tanker overturned spill blocking highway traffic',
        'Multiple cars piled up in fog on motorway',
        'Bus lost brakes and crashed into market area',
        'Accident on bridge causing traffic to stop completely',
        'Truck hit power pole lines down road blocked',
        'Vehicle fire after collision on GT road near city',
        'Road accident injured people need emergency help now',
        'Heavy vehicle overturned goods spilled on road',
        'Speeding car crashed into shop killing two people',
        'Rickshaw and motorcycle collision serious injuries',
        'School van accident children injured taken to hospital',
        'Driver fell asleep vehicle went off road into field',
        'Train and truck collision at level crossing fatalities',
        'Hit and run accident victim lying on road needs help',
        'Bus plunged into river after bridge railing broke',
        'Accident caused by wrong way driver on motorway',
        'Motorcycle rider hit by car pedestrian also injured',
    ]
}

rows = []
for label, texts in DESCRIPTIONS.items():
    for text in texts:
        rows.append({'text': text, 'label': label, 'source': 'synthetic'})

random.shuffle(rows)
df_synthetic = pd.DataFrame(rows)

csv_path = os.path.join(OUTPUT_DIR, 'text_descriptions.csv')
df_synthetic.to_csv(csv_path, index=False)

print(f'Generated {len(df_synthetic)} synthetic text descriptions')
print()
print('Per class:')
print(df_synthetic['label'].value_counts().to_string())
print(f'\nSaved to: {csv_path}')

Generated 100 synthetic text descriptions

Per class:
label
flood               25
traffic_incident    25
earthquake          25
fire                25

Saved to: /content/drive/MyDrive/crisis connect_phase3a/text_descriptions.csv


## Cell 5 — Transcribe Urdu/code-switched audio using Whisper
This cell is now ACTIVE.
Whisper transcribes Urdu and code-switched speech.
deep-translator then translates to English for TF-IDF.

In [ ]:
import os
import pandas as pd
from deep_translator import GoogleTranslator

AUDIO_EXTS = ('.mp3', '.wav', '.m4a', '.ogg', '.flac', '.webm')
transcribed_rows = []

if not os.path.exists(AUDIO_DIR):
    print('Audio folder not found at:', AUDIO_DIR)
    print('Create crisis connect_audio/ on Drive with subfolders and re-run.')
else:
    print('Starting audio transcription...')
    print('Whisper will auto-detect Urdu/English. Urdu will be translated to English.')
    print()

    for cls in CLASSES:
        cls_audio_path = os.path.join(AUDIO_DIR, cls + ' audio')
        if not os.path.exists(cls_audio_path):
            print(f'  {cls} audio/ folder not found, skipping')
            continue

        audio_files = [f for f in os.listdir(cls_audio_path)
                       if f.lower().endswith(AUDIO_EXTS)]
        print(f'  {cls}: {len(audio_files)} audio files')

        for fname in audio_files:
            fpath = os.path.join(cls_audio_path, fname)
            try:
                # Transcribe — Whisper auto detects language
                result       = whisper_model.transcribe(fpath)
                raw_text     = result['text'].strip()
                detected_lang = result.get('language', 'en')

                # Translate to English if Urdu or any non-English detected
                if detected_lang != 'en':
                    try:
                        english_text = GoogleTranslator(
                            source='auto',
                            target='en'
                        ).translate(raw_text)
                    except Exception:
                        english_text = raw_text  # fallback to original
                else:
                    english_text = raw_text

                transcribed_rows.append({
                    'text':     english_text,
                    'label':    cls,
                    'source':   'audio',
                    'original': raw_text,
                    'lang':     detected_lang
                })

                print(f'    [{detected_lang}] {fname}')
                print(f'          Original  : "{raw_text[:70]}"')
                print(f'          Translated: "{english_text[:70]}"')
                print()

            except Exception as e:
                print(f'    ERROR on {fname}: {e}')

    if transcribed_rows:
        audio_df   = pd.DataFrame(transcribed_rows)
        audio_path = os.path.join(OUTPUT_DIR, 'audio_transcriptions.csv')
        audio_df.to_csv(audio_path, index=False)
        print(f'Transcribed {len(transcribed_rows)} audio clips.')
        print(f'Saved to: {audio_path}')
        print()
        print('Per class breakdown:')
        print(audio_df['label'].value_counts().to_string())
    else:
        print('No audio transcribed. Check your audio folders on Drive.')

Starting audio transcription...
Whisper will auto-detect Urdu/English. Urdu will be translated to English.

  earthquake: 6 audio files
    [ur] New Recording.m4a
          Original  : "زرزلا ہے دیوارے ٹوٹ گئی ہیں لوگ بھاگ رہے ہیں"
          Translated: "There is an earthquake, the walls are broken, people are running away"

    [hi] New Recording 2.m4a
          Original  : "जलजला आए बिल्टिंग पूरी निचे गर्गे लोग नदर पश्वें प्लीज आप ख़ो कोई"
          Translated: "Jaljala aaye building puri niche garge log nadar pshwane please aap ko"

    [hi] New Recording 3.m4a
          Original  : "बगड़नाग ज़द रहा है, जमीन तीसे किन तक खिलती रहे है, गैस के बदबू भी आरी "
          Translated: "The sound of a bug is rising, the land has been blooming till now, the"

    [ur] New Recording 4.m4a
          Original  : "زلزلہ آئر اس کے افترشہ اکسارے کے بلدنگ میں واپس جانے سے لوگ دیر رہے ہی"
          Translated: "After the earthquake, people are staying late due to Aksare's return t"

    [hi] New Recor

## Cell 6 — Merge audio transcriptions with synthetic text and clean

In [ ]:
import re
import pandas as pd
import os
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
KEEP_WORDS = {'no', 'not', 'very', 'too', 'above', 'below', 'near', 'under', 'over'}
stop_words = stop_words - KEEP_WORDS

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [t for t in text.split() if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

# Load synthetic
df = pd.read_csv(os.path.join(OUTPUT_DIR, 'text_descriptions.csv'))
print(f'Synthetic descriptions: {len(df)}')

# Merge audio transcriptions
audio_path = os.path.join(OUTPUT_DIR, 'audio_transcriptions.csv')
if os.path.exists(audio_path):
    audio_df = pd.read_csv(audio_path)
    df = pd.concat([df, audio_df[['text', 'label', 'source']]], ignore_index=True)
    print(f'Audio transcriptions merged: {len(audio_df)}')
else:
    print('No audio transcriptions found.')

# Clean
df['cleaned_text'] = df['text'].apply(clean_text)
df = df[df['cleaned_text'].str.len() > 5].reset_index(drop=True)

print()
print(f'Total samples after merge and cleaning: {len(df)}')
print()
print('Per class:')
print(df['label'].value_counts().to_string())
print()
print('Per source:')
print(df['source'].value_counts().to_string())
print()
print('Sample cleaned texts:')
for _, row in df.sample(4, random_state=0).iterrows():
    print(f'  [{row["label"]}] [{row["source"]}]  {row["cleaned_text"][:80]}')

Synthetic descriptions: 100
Audio transcriptions merged: 24

Total samples after merge and cleaning: 124

Per class:
label
flood               31
traffic_incident    31
earthquake          31
fire                31

Per source:
source
synthetic    100
audio         24

Sample cleaned texts:
  [earthquake] [synthetic]  power lines fell strong earthquake tremor
  [traffic_incident] [synthetic]  motorcycle rider hit car pedestrian also injured
  [flood] [synthetic]  water rising every hour need immediate evacuation help
  [traffic_incident] [synthetic]  traffic jam due accident ambulance cannot reach site


## Cell 7 — TF-IDF Vectorisation

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
import numpy as np

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=3000,
    min_df=1,
    sublinear_tf=True
)

X = vectorizer.fit_transform(df['cleaned_text'])

le = LabelEncoder()
le.fit(CLASSES)
y = le.transform(df['label'])

print(f'TF-IDF matrix shape : {X.shape}')
print(f'  Rows = {X.shape[0]} text samples')
print(f'  Cols = {X.shape[1]} TF-IDF features')
print()
print('Label encoding:')
for i, cls in enumerate(le.classes_):
    print(f'  {i} -> {cls}')
print()
print('Top 10 TF-IDF features per class:')
feature_names = vectorizer.get_feature_names_out()
for cls in CLASSES:
    cls_idx   = le.transform([cls])[0]
    cls_mask  = y == cls_idx
    cls_tfidf = np.asarray(X[cls_mask].mean(axis=0)).flatten()
    top_idx   = cls_tfidf.argsort()[-10:][::-1]
    top_words = [feature_names[i] for i in top_idx]
    print(f'  {cls:<22} {top_words}')

TF-IDF matrix shape : (124, 1208)
  Rows = 124 text samples
  Cols = 1208 TF-IDF features

Label encoding:
  0 -> earthquake
  1 -> fire
  2 -> flood
  3 -> traffic_incident

Top 10 TF-IDF features per class:
  earthquake             ['earthquake', 'collapsed', 'strong', 'people', 'tremor', 'quake', 'walls', 'shaking', 'felt', 'building']
  flood                  ['water', 'flood', 'village', 'submerged', 'area', 'rising', 'main', 'near', 'flooded', 'flood water']
  fire                   ['fire', 'burning', 'house', 'spreading', 'building', 'smoke', 'house fire', 'inside', 'forest', 'market']
  traffic_incident       ['road', 'accident', 'injured', 'collision', 'hit', 'vehicle', 'people', 'traffic', 'car', 'bus']


## Cell 8 — Split and save all files for  (Phase 3b)

In [ ]:
from sklearn.model_selection import train_test_split
import scipy.sparse as sp
import joblib, os
import numpy as np

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'Train : {X_train.shape[0]} samples')
print(f'Val   : {X_val.shape[0]} samples')
print(f'Test  : {X_test.shape[0]} samples')
print()

sp.save_npz(os.path.join(OUTPUT_DIR, 'X_train.npz'), X_train)
sp.save_npz(os.path.join(OUTPUT_DIR, 'X_val.npz'),   X_val)
sp.save_npz(os.path.join(OUTPUT_DIR, 'X_test.npz'),  X_test)
np.save(os.path.join(OUTPUT_DIR, 'y_train.npy'), y_train)
np.save(os.path.join(OUTPUT_DIR, 'y_val.npy'),   y_val)
np.save(os.path.join(OUTPUT_DIR, 'y_test.npy'),  y_test)
joblib.dump(vectorizer, os.path.join(OUTPUT_DIR, 'tfidf_vectorizer.pkl'))
joblib.dump(le,         os.path.join(OUTPUT_DIR, 'label_encoder.pkl'))
df.to_csv(os.path.join(OUTPUT_DIR, 'cleaned_texts.csv'), index=False)

print('All files saved to crisis connect_phase3a/ on Drive:')
print()
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024
    print(f'  {f:<35} {size:.1f} KB')

Train : 86 samples
Val   : 19 samples
Test  : 19 samples

All files saved to crisis connect_phase3a/ on Drive:

  X_test.npz                          2.5 KB
  X_train.npz                         7.3 KB
  X_val.npz                           2.4 KB
  audio_transcriptions.csv            11.0 KB
  cleaned_texts.csv                   17.2 KB
  label_encoder.pkl                   0.6 KB
  text_descriptions.csv               7.0 KB
  tfidf_vectorizer.pkl                48.4 KB
  y_test.npy                          0.3 KB
  y_train.npy                         0.8 KB
  y_val.npy                           0.3 KB


## Cell 9 — Summary

In [ ]:
import os

print('=' * 55)
print('  Phase 3a Complete — Audio + Text Preprocessing')
print('=' * 55)
print()
print(f'Synthetic descriptions : 100')
audio_path = os.path.join(OUTPUT_DIR, 'audio_transcriptions.csv')
if os.path.exists(audio_path):
    import pandas as pd
    adf = pd.read_csv(audio_path)
    print(f'Audio transcriptions   : {len(adf)}')
    print(f'Total training samples : {100 + len(adf)}')
else:
    print('Audio transcriptions   : 0')
print()
print('Files saved to: crisis connect_phase3a/')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024
    print(f'  {f:<35} {size:.1f} KB')
print()
print('Next steps:')
print('   -> re-run Phase 3b with updated TF-IDF files')
print('   -> re-run Phase 4 with updated models')
print('  Check if accuracy improved after adding audio data')

  Phase 3a Complete — Audio + Text Preprocessing

Synthetic descriptions : 100
Audio transcriptions   : 24
Total training samples : 124

Files saved to: crisis connect_phase3a/
  X_test.npz                          2.5 KB
  X_train.npz                         7.3 KB
  X_val.npz                           2.4 KB
  audio_transcriptions.csv            11.0 KB
  cleaned_texts.csv                   17.2 KB
  label_encoder.pkl                   0.6 KB
  text_descriptions.csv               7.0 KB
  tfidf_vectorizer.pkl                48.4 KB
  y_test.npy                          0.3 KB
  y_train.npy                         0.8 KB
  y_val.npy                           0.3 KB

Next steps:
   -> re-run Phase 3b with updated TF-IDF files
   -> re-run Phase 4 with updated models
  Check if accuracy improved after adding audio data
